# 03 · Maps in Python
1. Plot the **observed flood maps** and the **training points** (files exported from GEE)
2. Make a **flood susceptibility map** by applying the saved Random Forest to every pixel of a
   factor stack (`data/stacks/<area>_factor_stack.tif`, exported by `00_gee_data_prep.ipynb`)
3. Change the rainfall to make **scenario maps** (the prediction-system idea)

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import rasterio
import geopandas as gpd

from src.config import DATA, OUT, EVENTS, GEE_COLUMNS, CLASS_NAMES, CLASS_COLORS
from src.ml_utils import add_features

## 1 · Observed flood maps + training points

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, ev in zip(axes, EVENTS):
    tif = DATA / 'flood_maps' / f"{ev['name']}_flood_map.tif"
    shp = DATA / 'flood_maps' / f"{ev['name']}_flood_points.shp"
    if not tif.exists():
        ax.set_title(f"{ev['name']}: file missing"); ax.axis('off'); continue
    with rasterio.open(tif) as src:
        img = src.read(1).astype(float)
        bounds, crs = src.bounds, src.crs
    img[img == 255] = np.nan
    ax.imshow(img, cmap=ListedColormap(['#f2f2f2', '#1f78b4']),
              extent=[bounds.left, bounds.right, bounds.bottom, bounds.top])
    if shp.exists():
        pts = gpd.read_file(shp).to_crs(crs)
        pts[pts['label'] == 0].plot(ax=ax, color='green', markersize=1, label='Non-flood')
        pts[pts['label'] == 1].plot(ax=ax, color='red', markersize=1, label='Flood')
        ax.legend(loc='lower right', markerscale=8)
    ax.set_title(ev['name']); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.savefig(OUT / 'observed_flood_maps.png', dpi=200); plt.show()

## 2 · Susceptibility map from a factor stack
`RAIN_EVENT` / `RAIN_ANTEC` = `None` keeps the rainfall stored in the stack;
set numbers (mm) to try a different storm.

In [ ]:
AREA = 'hyderabad'            # file: data/stacks/hyderabad_factor_stack.tif
MODEL = 'rf_no_elevation'
RAIN_EVENT, RAIN_ANTEC = None, None     # e.g. 230, 450 for an extreme storm

pipe = joblib.load(OUT / f'{MODEL}.pkl')
FEATURES = json.loads((OUT / 'model_features.json').read_text())[MODEL]

with rasterio.open(DATA / 'stacks' / f'{AREA}_factor_stack.tif') as src:
    arr = src.read().astype('float32')            # (14 bands, rows, cols)
    profile = src.profile
bands, rows, cols = arr.shape
table = pd.DataFrame(arr.reshape(bands, -1).T, columns=GEE_COLUMNS)
if RAIN_EVENT is not None:
    table['rain_event'] = RAIN_EVENT
if RAIN_ANTEC is not None:
    table['rain_antec'] = RAIN_ANTEC
table = add_features(table)

valid = table[FEATURES].notna().all(axis=1).to_numpy()
prob = np.full(len(table), np.nan, dtype='float32')
idx = np.flatnonzero(valid)
for start in range(0, len(idx), 500_000):           # predict in chunks to save memory
    part = idx[start:start + 500_000]
    prob[part] = pipe.predict_proba(table.iloc[part][FEATURES])[:, 1]
prob = prob.reshape(rows, cols)
classes = np.where(np.isnan(prob), 0, np.minimum(np.floor(prob * 5), 4) + 1).astype('uint8')
print(f'{AREA}: {valid.sum():,} pixels predicted')

In [ ]:
tag = AREA if RAIN_EVENT is None else f'{AREA}_rain{RAIN_EVENT}'
profile.update(count=1, dtype='float32', nodata=np.nan, compress='lzw')
with rasterio.open(OUT / f'{tag}_flood_probability.tif', 'w', **profile) as dst:
    dst.write(prob, 1)
profile.update(dtype='uint8', nodata=0)
with rasterio.open(OUT / f'{tag}_susceptibility_classes.tif', 'w', **profile) as dst:
    dst.write(classes, 1)

fig, ax = plt.subplots(figsize=(10, 8))
shown = np.ma.masked_equal(classes, 0)
ax.imshow(shown, cmap=ListedColormap(CLASS_COLORS), vmin=1, vmax=5)
ax.legend(handles=[Patch(color=c, label=n) for c, n in zip(CLASS_COLORS, CLASS_NAMES)],
          loc='lower left', title='Flood susceptibility')
ax.set_title(f'Flood susceptibility — {tag}'); ax.axis('off')
plt.tight_layout(); plt.savefig(OUT / f'{tag}_susceptibility_map.png', dpi=200); plt.show()

In [ ]:
# Area per class
pixel_km2 = abs(profile['transform'].a * profile['transform'].e) / 1e6
counts = pd.Series(classes[classes > 0]).value_counts().sort_index()
area = pd.DataFrame({'Class': [CLASS_NAMES[i - 1] for i in counts.index],
                     'Area (km²)': (counts.values * pixel_km2).round(1)})
area['Share (%)'] = (100 * area['Area (km²)'] / area['Area (km²)'].sum()).round(1)
area.to_csv(OUT / f'{tag}_area_per_class.csv', index=False)
area

## 3 · Rainfall scenarios (prediction-system idea)
Same area, three storms. Uses the variables from section 2.

In [ ]:
SCENARIOS = {'Light (100 mm)': (100, 180), 'Typical (150 mm)': (150, 265), 'Extreme (230 mm)': (230, 450)}

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
base = add_features(pd.DataFrame(arr.reshape(bands, -1).T, columns=GEE_COLUMNS))
for ax, (label, (r_ev, r_an)) in zip(axes, SCENARIOS.items()):
    t = base.copy()
    t['rain_event'], t['rain_antec'] = r_ev, r_an
    p = np.full(len(t), np.nan, dtype='float32')
    for start in range(0, len(idx), 500_000):
        part = idx[start:start + 500_000]
        p[part] = pipe.predict_proba(t.iloc[part][FEATURES])[:, 1]
    c = np.where(np.isnan(p), 0, np.minimum(np.floor(p * 5), 4) + 1).reshape(rows, cols)
    high = np.isin(c, [4, 5]).sum() / (c > 0).sum() * 100
    ax.imshow(np.ma.masked_equal(c, 0), cmap=ListedColormap(CLASS_COLORS), vmin=1, vmax=5)
    ax.set_title(f'{label}\nHigh + very high: {high:.1f}% of area'); ax.axis('off')
fig.legend(handles=[Patch(color=c, label=n) for c, n in zip(CLASS_COLORS, CLASS_NAMES)],
           loc='lower center', ncol=5)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig(OUT / f'{AREA}_rainfall_scenarios.png', dpi=200); plt.show()